# __Dream interpretation RAG__

## __Installs & imports__

In [1]:
! pip install chromadb sentence-transformers

In [36]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import re
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

## __Data preparation__

In [44]:
data = pd.read_csv('/Users/lesfabs/code/MartynaC/dreamscope/Notebooks/dream_symbols_clean_v2.csv') # Use version 5
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are ...
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more ...
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead o...
3,macadamize,macadamize,"To see, walk or travel on a macadamized road i...",to see walk or travel on a macadamized road in...,suggests that you are standing on solid ground...
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be t...


In [3]:
data.context_clean.isnull().sum()

np.int64(7)

In [4]:
# Replace NaNs in  context_clean with the value of the slug column 
data.loc[data['context_clean'].isnull(), 'context_clean'] = data.loc[data['context_clean'].isnull(), 'slug']
data.context_clean.isnull().sum()

np.int64(0)

In [5]:
# Create chunks
data['chunk'] = data['context_clean'] + " " + data['meaning_clean']
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are ...,to see the letter m in your dream suggests tha...
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more ...,to see or eat mms in your dream symbolizes lif...
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead o...,to dream that you are a giant talking mm sugge...
3,macadamize,macadamize,"To see, walk or travel on a macadamized road i...",to see walk or travel on a macadamized road in...,suggests that you are standing on solid ground...,to see walk or travel on a macadamized road in...
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be t...,to dream that you are eating macaroni symboliz...


## __MVP__

### __Embedding__

In [6]:
sentence_transformer_model = SentenceTransformer("all-mpnet-base-v2", device = 'mps')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# One chunk
tryout_vector = sentence_transformer_model.encode(data['chunk'][0])
tryout_vector.shape

(768,)

In [8]:
# All chunks

embeddings = sentence_transformer_model.encode(data['chunk'], show_progress_bar=True)


Batches:   0%|          | 0/244 [00:00<?, ?it/s]

In [9]:
embeddings.shape

(7798, 768)

### __Setting up and querying collection__

In [10]:
# Initializing collection - slightly different from the method seen in previous challenges using LangChain 
# -> Here it's native ChromaDB, without an intermediary framework
client = chromadb.Client()

In [49]:
# To delete the collection if needed
#client.delete_collection("dream_symbols")

In [11]:
collection = client.create_collection("dream_symbols")

In [12]:
# Adding and indexing chunks

collection.add(
    documents=data['chunk'].tolist()[:5000],
    embeddings=embeddings[:5000],
    ids=[str(i) for i in range(5000)]
)



In [13]:
# We do it in two batches because of the limited batch size

collection.add(
    documents=data['chunk'].tolist()[5000:],
    embeddings=embeddings[5000:],
    ids=[str(i) for i in range(5000, 7798)]
)


In [14]:
collection.count()

7798

In [46]:
dreamer_text = "I was trapped in a job for some company that was making quantum computers based on slime molds - in some small hangars outside the city, they had a room, a secret one, where there were slime molds and computers and some kind of turbo technology. I was supposed to work for them, but I knew that the two guys running it were dangerous fools and I was trying to reach people to raise awareness that something like this was happening there and that I was somehow involved in it and that it needed to be stopped and something better done with this technology."

In [47]:
searched_vector = sentence_transformer_model.encode(dreamer_text)
searched_vector.shape

(768,)

In [17]:
result=collection.query(
    searched_vector,
    n_results=6
)
result['documents']

[['to dream that you are riding a giraffe represents your desire to stand up amongst the crowd you want attention but arent getting it',
  'to dream that you are eating macaroni symbolizes comfort and ease the dream may be trying to bring you back to a time where things were much simpler',
  'to see a giraffe in your dream suggests that you need to consider the overall picture take a broader view on your life and where it is headed the dream may also be a metaphor on how you are sticking your neck out for someone',
  'to see or eat a macaroon in your dream represents a wellrounded experience you are successfully balancing different aspects of your life',
  'to dream that a giraffe is running implies that you are avoiding the truth it also suggests that you saw something that you shouldnt have seen',
  'to dream that you are on a safari represents freedom from societal norms and rules you are trying to break free from the confines of civilization']]

### __Compiling query process in a dedicated function__ 

In [18]:
# Compile code in a dedicated function

def interpret(dream='',n_results=3):
    searched_vector = sentence_transformer_model.encode(dream)
    result=collection.query(
    searched_vector,
    n_results=n_results
    )
    return result['documents']

interpret(dreamer_text)

[['to dream that you are riding a giraffe represents your desire to stand up amongst the crowd you want attention but arent getting it',
  'to dream that you are eating macaroni symbolizes comfort and ease the dream may be trying to bring you back to a time where things were much simpler',
  'to see a giraffe in your dream suggests that you need to consider the overall picture take a broader view on your life and where it is headed the dream may also be a metaphor on how you are sticking your neck out for someone']]

## __Trying to fine-tune the querying process__

### __Extracting keywords - Simple version__

In [48]:
# Initialize Model

tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

# Initialize pipeline

pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    )


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [51]:
# Define function to extract keywords

def extract_keywords(dreamer_text, n_keywords):
    # define prompt template
    prompt = f'You process a text input by extracting the {n_keywords} most significant keywords, ranked by significance. Expected format : a Python list of singular words, without any additional text'
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': dreamer_text}
    ]

    # query model
    output = pipe(messages) 
    return output[0]['generated_text'][-1]['content']
    

In [52]:
result = extract_keywords(dreamer_text, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [53]:
result

' ["quantum", "slime molds", "computers", "turbo technology", "dangerous", "awareness"]'

In [54]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)


In [55]:
keywords

['quantum',
 'slime molds',
 'computers',
 'turbo technology',
 'dangerous',
 'awareness']

In [56]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result=collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

quantum ['refbeating refbeating']
slime molds ['if there is mold growing on your food then it implies that you are too easily susceptible to some negative energy around you']
computers ['refcopier refcopier']
turbo technology ['to see an accelerator in your dream indicates that you will achieve your goals through your own efforts the dream may also be telling you to slow down']
dangerous ['to see dangerous people in your dream suggests that you need to proceed with caution in some endeavor']
awareness ['dreaming that you are watching or are at a press conference refers to your expanded awareness']


### __Extracting keywords - Advanced version with preprocessing__

In [37]:
# Improve function to extract keywords -> Preprocess dream text

stop_words = set(stopwords.words('english'))

def preprocess(dream_text):
    # remove punctuation
    for punctuation in string.punctuation:
        dream_text = dream_text.replace(punctuation, ' ')

    # lowercase
    dream_text = dream_text.lower()
    
    # remove stopwords
    tokenized_text = word_tokenize(dream_text)
    clean_text = [word for word in tokenized_text if not word in stop_words]

    # lemmatize
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in clean_text]
    lemmatized_string = " ".join(lemmatized)
    
    return lemmatized_string


In [57]:
preprocessed_dream = preprocess(dreamer_text)
preprocessed_dream

'trapped job company making quantum computer based slime mold small hangar outside city room secret one slime mold computer kind turbo technology supposed work knew two guy running dangerous fool trying reach people raise awareness something like happening somehow involved needed stopped something better done technology'

In [58]:
result = extract_keywords(preprocessed_dream, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [59]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)
keywords

['quantum computer',
 'slime mold',
 'turbo technology',
 'dangerous',
 'awareness',
 'stop']

In [60]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result=collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

quantum computer ['to see a computer in your dream symbolizes technology information and modern life new areas of opportunities are being opened to you alternatively computers represent a lack of individuality and lack of emotions and feelings too often you are just going along with the flow without voicing your own opinions and views you may also feel a depreciated sense of superiority']
slime mold ['refspine refspine']
turbo technology ['to see an accelerator in your dream indicates that you will achieve your goals through your own efforts the dream may also be telling you to slow down']
dangerous ['to see dangerous people in your dream suggests that you need to proceed with caution in some endeavor']
awareness ['dreaming that you are watching or are at a press conference refers to your expanded awareness']
stop ['refannouncer refannouncer']


In [ ]:
## TODO
# Try BerTopic https://maartengr.github.io/BERTopic/index.html // First out of the box, then trained on the symbols dictionary.
# Also try Named Entity Recognition (gleaner) to extract the most significant words from the text.
# Generate final response with LLM 